# §1 Introduction

### 1.1 Aims

This report aims to investigate N-body gravitational simulations in relation to virialisation. Virialisation is the process where a gravitationally bound system attains a steady state where

$$
2T + U=0, \tag{1}
\\
\\
\frac{2T}{|U|} = 1. 
$$
With T being the total kinetic energy and U as the total gravitational potential energy **[1]**. This is a key process in physics, as it explains how dark matter halos ceased gravitational collapse and establish the underlying potential structure that enables galaxy formation **[2]**. Consequently, it is important to develop algorithms that accurately reproduce the Virial theorem when performing cosmological simulations.

In this report the time complexity of the pairwise, Barnes-Hut (BH) and fast multipole method (FMM) algorithms have been investigated for up to $10^{3}$ particles in regimes of $r_{virial} = 10^{-3}, 10^{0}, 10^{3}$. The physical accuracy of these systems was probed via observing momentum and energy conservation in these same regimes.

### 1.2 History

Computers became powerful enough in the mid 1980s for cosmologists to start running N-body gravity simulations. One of the best known by G. Efstathiou and M. Davis successfully used a particle and mesh-based algorithm to simulate ~32,000 particles **[3]**. The accuracy and scale of the programs has steadily increased over time and modern simulations like the 2022 MillenniumTNG Project simulated galactic stuctures in volumes of $(740Mpc)^{3}$ **[4]**.

Both of these projects were written in machine code or highly optimised C and were run on state-of-the-art supercomputers. By contrast, this project was written in Python on a personal laptop, and therefore has a smaller scope and much lower N.

# §2 Algorithms

### 2.2 Pairwise Algorithm

This is the simplest acceleration calculation algorithm, and operates by looping over every particle and calculating the force on that particle caused by all other particles.
The pseudocode is as follows:


```text

BEGIN
FOR i = 1 to N:
    SET  a_{i}= 0
    FOR j = 1 to N:
        IF i ≠ j:
            CALCULATE   a_{ij}
            SET a_{i} = a_{i} + a_{ij}
END

```

**Fig. 1:** Pseudocode describing the pairwise N-body algorithm. Line 5 removes unphysical self-gravitation for point masses.

The nested for-loop implies an $O(N^{2})$ time complexity. Newton's third law may be used to half the calculations as $\vec{F_{ij}} = -\vec{F_{ji}}$ however this does not change the time complexity


### 2.3 Barnes-Hut (BH) Algorithm

The Barnes–Hut algorithm was introduced in 1986 by Josh Barnes and Piet Hut [5]. It builds on earlier hierarchical methods for computing gravitational interactions, such as Appel's algorithm, which organises particles into a tree structure to compute a potential surface which is then differentiated to find the force.[6]. (It was later found that under some conditions Appel's algorithm is actually O(N) [7].) The key innovation of BH is that it includes a $\theta$ parameter that allows easy tuning of accuracy vs computational overhead. 

The Barnes–Hut algorithm operates by partitioning a system of N particles into hierarchical groupings stored in an octree. Each node in the tree corresponds to a region of physical space. If a particle is in a node which already contains a particle, the node is recursively subdivided until each subdivision contains at most one particle. The tree consists of two node types: internal (root) nodes, which contain child nodes and no particles, and external (leaf) nodes, which have no children but may contain a particle. The centre of mass (COM) and total mass of the particles in a node are calculated as the tree is constructed. The tree is completely rebuilt every time step.

To compute the acceleration on a single particle the tree is traversed starting from the root. If a node is sufficiently far from the particle, it is approximated as being a point mass positioned at the COM and with total mass of the node. This is a dipole approximation. If the node is sufficiently close to the particle the process is repeated for the node's children. 

A node is considered sufficiently distant if the ratio of its width to the separation between its centre of mass and the particle of interest is less than a tunable threshold $\theta$, i.e.

$$
\frac{s}{r} < \theta, \tag{5}
$$

where s is the node size and r is the distance to its centre of mass. $\theta$ affects the accuracy of the simulation, large values of $\theta \sim 1$ reduce runtime but also accuracy. If $\theta = 0$, no approximations are made and BH reduces to the pairwise algorithm.




```text
BEGIN
FUNCTION create_node(centre, width):
    SET node centre = centre
    SET node width = width
    SET node particle  = none
    SET node mass = 0
    SET node COM = [0,0,0]
    SET node children = none
```


**Fig. 2:** Pseudocode for initialising a node. Note that "node centre" and "node COM" are vector quantities.

```text
BEGIN
FUNCTION create_node(centre, width):
    SET node centre = centre
    SET node width = width
    SET node particle  = none
    SET node mass = 0
    SET node COM = [0,0,0]
    SET node children = none

FUNCTION insert(node, particle):
    IF particle is not inside node:
        RETURN
    SET old mass = node mass
    SET node mass = old mass + particle mass
    SET new COM = (node COM * old mass + particle position * particle mass) / node mass
    SET node COM = new COM
    IF node children = none AND node particle = none:
        SET node particle = particle
        RETURN
    IF node children = none:
        FOR i = 1 to 8:
            SET child = create_node(appropriate centre, node width / 2)
            ADD child to node children
        FOR i in node children:
            CALL insert(i, node particle)
        SET node particle = none
    FOR i in node children:
        CALL insert(i, particle)

```

**Fig. 3:** Pseudocode for inserting a particle into a node. First it checks to see if the particle is physically within the node. Then it updates the mass and COM of the node. After this it checks to see if the node already contains a particle. If so, 8 equal sized child nodes are produced and both particles are placed in them. This is recursively done untill every particle is in its own node. Note that root nodes only have children, they do not contain particles.

```text
FUNCTION acceleration(particle, node, θ):
    IF node particle = particle
        RETURN
    SET separation = |node COM – particle position|
    IF node children = none:
        CALCULATE a_{node}
        SET a_{particle} = a_{particle} + a_{node}
        RETURN
    IF node width / separation < θ :
          CALCULATE a_{node}
        SET a_{particle} = a_{particle} + a_{node}
        RETURN
    FOR child in node children:
        CALL acceleration(particle, child, θ)
```

**Fig. 4:** Tree traversal for acceleration calculation. Self-interactions are excluded, leaf nodes are evaluated directly, and internal nodes are either approximated using their COM and mass or recursively opened depending on the $\theta$ criterion. Note that the leaf nodes are not approximated as their node properties are identical to the particle within (should they contain one).

```text
FUNCTION system_acceleration(particles, θ):
    SET root = create_node(system centre, system width)
    FOR particle in particles:
        CALL insert(root, particle)
    FOR particle in particles:
        SET  a_{particle} = [0,0,0]
        CALL acceleration(particle, root, θ)
END
```

**Fig. 5**: Calculation of particle accelerations. The first loop constructs the octree and inserts the particles. The second loop computes the acceleration of each particle via tree traversal. The acceleration $\vec{a_{particle}}$ is reset to zero at each timestep, as forces are totally recomputed rather than updated from the previous step.


To estimate the time complexity, one must consider the depth of the tree. For a roughly uniform particle distribution, the mean separation scales as $\Delta x \sim L/N^{1/3}$, where $L$ is the system size. Since leaf nodes contain at most one particle, their width is $\sim \Delta x$. Each subdivision halves the cell size, so the mean tree depth is $\sim \log_{2}(N) / 3$
implying $O(\log N)$ work per traversal. Inserting N particles therefore requires $O(N\log (N))$ operations. 

The time complexity of the acceleration calculation can be understood by considering the work required for a single particle. For a homogeneous distribution, increasing the number of particles by a factor of eight corresponds to adding one additional level to the tree (it is like joining eight similar nodes together). This increases the number of node interactions by a constant amount (dependent on $\theta$, but independent of $N$). Thus, the work per particle scales as $O(\log N)$, implying a total cost of $O(N \log N)$ to compute all accelerations.

As both tree construction and force evaluation scale as $O(N \log N)$, the overall algorithm has the same complexity.


### 2.3 Fast Multipole Method (FMM) Algorithm
The Fast Multipole Method (FMM) was introduced in 1987 by Leslie Greengard and Vladimir Rokhlin [8]. Like the Barnes–Hut algorithm, it employs a hierarchical octree to organise particles, but extends this approach by allowing node-node interactions via multipole expansions. This is the most complicated of the three algorithms.

First the space is divided into an octree of depth $\log(N)$. This is a perfect tree - independent of particle distribution and every root node has eight children except leaf nodes. A parameter $P$ sets the order of the truncated expansions about each node’s centre. Each node carries two representations: a multipole potential expansion describing the contribution of masses within the node, and a local potential expansion representing the influence of distant masses. In addition, each node maintains an interaction list consisting of well-separated nodes at the same level.

An upward pass begins at the leaves, where multipole expansions are computed from the enclosed particles. These are then aggregated up the tree, with each parent forming its multipole expansion from those of its children, continuing recursively to the root.

A downward pass then propagates information from the root to the leaves. For each node, a local expansion is formed by combining contributions from the multipole expansions of nodes in its interaction list with the local expansion inherited from its parent. This process continues to the leaf nodes.

Particle accelerations are computed by evaluating direct interactions between particles within the same or neighbouring leaf nodes. The local expansions are then evaluated at the particle positions, and their contributions are combined with the direct interactions to obtain the total acceleration.

The local expansion attempts to approximate the potential at a particle due to masses at the primed positions. It works the following way:

$$
\phi = \sum_{i} \frac{m_{i}}{|\vec{x} - \vec{x_{i}}^{\prime}|}. \tag{6}
$$

We then define $\vec{r} = \vec{c_{t}} - \vec{x}$, $\vec{r^{\prime}} = \vec{c_{t}} - \vec{x^{\prime}}$, and $\vec{R} = \vec{_{t}} - \vec{c_{s}}$. $\vec{c_{t}}$ is the centre of the node containing the target point, and $\vec{c_{s}}$ is the centre of the node containing the sourcing points. This is written as

$$
\phi = \sum_{i} \frac{m_{i}}{|\vec{R} - \vec{r_{i}}^{\prime}|}. \tag{7}
$$

A multivariate Taylor expansion can then be taken of the distance term of the potential:

$$
\frac{1}{|\vec{R} - \vec{r^{\prime}}|} = \sum_{n=0}^{p \rightarrow  \infin} \frac{(-1)^{n}}{n!} (\vec{r^{\prime}} \cdot \nabla)^{n} \left [\frac{1}{|\vec{R}|} \right]. \tag{8}
$$

Recombining with the masses this gives:

$$
\phi = \sum_{n=0}^{p \rightarrow \infin} \frac{(-1)^{n}}{n!} \left \{ \sum_{i} m_{i} (\vec{r_{i}^{\prime}})^{n} \right \} \nabla^{n} \left[\frac{1}{|\vec{R}|} \right]. \tag{9}
$$

The term within the braces represents the multipole expansion coefficients of the mass distribution of the source node. $P$ is the order of the approximation.

The length of the algorithm leads to a significant amount of pseudocode. In the pursuit of readability, the version presented here is intentionally less explicit than the BH pseudocode. 



```text
BEGIN
FUNCTION create_node():
    SET node centre 
    SET node width 
    SET node particles 
    SET node multipole_expansion 
    SET node local_expansion 
    SET node children
    SET node parents
    SET node siblings
    SET node cousins
    SET node interaction list
```


**Fig. 6:** Creation of a FMM node. Note that it is different from a BH node as it now contains 2 expansions and an interaction list. Note also that the cousin and sibling lists are present as they are used in the construction of the interaction lists.

```text
FUNCTION construct_tree(particle_system, max_depth):
    SET root_node = create_node(particle_system width)
    FOR i =1 to max_depth:
        FOR all nodes at depth = i:
            FOR j = 1 to 8:
            SET child = create_node(appropriate centre, node width /2):
            ADD child to node.children
    RETURN root_node = tree
```

**Fig. 7:** Construction of the tree. A "root_node" is constructed which bounds all particles. Then the node is recursively divided until the tree is the correct depth. Node positions, widths, children and parents are set at this stage.


```text
FUNCTION fill_tree(particle_system, tree):
    FOR particle in particle_system:
        INSERT particle into leaf node of tree
    FOR node in tree:
        FILL node interaction_list
```

**Fig. 8:** This acts on a constructed tree, adding the particles to the leaf node interaction lists and filling in the node cousins, siblings and interaction lists. The tree now contains the information required for the upwards pass.

```text
FUNCTION upwards_pass(tree, P):
    FOR leaf node in tree:
        FOR particle in leaf node:
            CALCULATE multipole expansion contribution of order P
            ADD multipole expansion to leaf node multipole_expansion
    FOR i in range max_depth to 1:
        FOR nodes as depth i:
            IF node is NOT leaf node:
                FOR child in node children:
                    SET multipole expansion contribution  = SHIFT(child multipole_expansion, node centre)
                    ADD multipole expansion contribution  to node multipole_expansion

```

**Fig. 9:** Upwards_path begins by iterating over the leaf nodes, computing the multipole expansion of order $P$ from each node's enclosed particles. This information is then propagated recursively up the tree, with parent nodes re-centering and aggregating the multipole expansions of their children.

```text
FUNCTION downwards pass(tree):
    FOR i = 1 to max_depth:
        FOR all node at depth i:
            SET node local_expansion = 0
            FOR source_node in node interaction_list:
                SET local contribution = SHIFT(source_node multipole_expansion, node centre)
                ADD local contribution to node local_expansion
        ADD node parent local_expansion to node local_expansion
    FOR all node at max depth:
        FOR particle in node particles:
            SET a_{particle} = [0,0,0]
            CALCULATE a_{local expansion} at particle
            SET a_{particle} = a_{local expansion}
            FOR j in node particles:
                IF j ≠ particle
                CALCULATE a_{j, particle}
                ADD a_{j, particle} to a_{particle}
            FOR sibling in node siblings:
                FOR j in sibling particles:
                    CALCULATE a_{j, particle}
                    ADD a_{j, particle} to a_{particle}
```

**Fig. 10:** "Downward_pass" computes the accelerations of the particles. The procedure begins at the root and recurses down the tree toward the leaves. At each node, the local expansion is inherited from its parent and augmented by contributions from nodes in its interaction list, whose multipole expansions are re-centered and converted into local (Taylor) expansions of the potential. This process continues until the leaf nodes are reached. At that stage, the acceleration due to the local field is evaluated for each particle. This is then combined with the contribution from direct particle–particle interactions within the same leaf node and its neighboring (sibling) leaf nodes.

```text
FUNCTION system_acceleration(particle_system, P, N):
	SET max_depth = ceiling(log_{8}(N))
	CALL fill_tree(particle_system, tree)
	CALL upwards_pass(tree, P)
	CALL downwards_pass(tree)
END
```

**Fig. 11:** This wrapper function calls the other functions in order to get the accelerations of every particle. Note how the max_depth is set to $\lceil \log_{8}(N) \rceil$ as this is important for the time complexity of the algorithm.



The time complexity of the FMM is $O(N)$ **[8]**. For a homogeneous particle distribution, an octree of depth $\lceil \log_{8}(N) \rceil$ contains $\sim N$ leaf nodes, so each leaf holds $O(1)$ particles on average.

In the upward pass, computing the multipole expansion at each leaf requires $O(1)$ work, giving a total cost of $O(N)$. Each internal node has a fixed number of children (eight), and aggregating their multipole expansions costs $O(p^2)$ due to the $p^2$ coefficients. Since the total number of nodes in the tree is $O(N)$, this stage also scales linearly.

For the downward pass, the interaction list has a maximum size of 189 in 3D. The localisation (multipole-to-local translation) depends on the expansion order $p$ but is independent of $N$, giving $O(1)$ work per cell and hence $O(N)$ overall. The final direct particle–particle interactions also cost $O(1)$ per cell, since each leaf contains $\sim 1$ particle and interacts only with a fixed number of neighbouring cells, independent of $N$.

Therefore, the computation of leaf multipole expansions, their upward propagation, the downward localisation, and the direct particle–particle interactions all scale linearly, resulting in an overall time complexity of $O(N)$ for the FMM.

# §3 Implementation
This simulation models all particles as having identical mass ($m=1$) as a simplification of the stellar mass spectrum. This is a reasonable approximation for studying bulk gravitational dynamics, since most of the stellar mass in typical galaxies is concentrated within sub-solar masses (roughly $0.1–1M_{\odot}$)
 and equal-mass models capture the leading-order behaviour of large-N self-gravitating systems.

The value of $G = 1$. This rescales the timescale over which the system evolves, $t_{dyn}$,to order unity.
$$
t_{dyn} = \sqrt{\frac{1}{G\rho}} \tag{2}
$$ 
where $\rho$ is the density. This enables a convenient choice of $dt = 10^{-3} - 10^{-2}$.


All the algorithms aim to simulate the following acceleration 
$$
\vec{a_{ij}} = \frac{-Gm_{i}m_{j}}{(r_{ij}^{2} + \epsilon_{ij}^{2})} \vec{\hat{r_{ij}}} \tag{3}
$$
where the symbols have their usual meanings and $\epsilon$ is the softening factor. The softening only exists to remove divisions by zero for particles in the same place.

### 3.1 Simulation Setup

The system is represented by an "NBodySystem" class, initialised with arrays of particle positions and velocities. These are stored internally as (N,3) NumPy arrays, where N is the number of particles. The class exposes these as ".positions" and ".velocities", and defines ".N" as the total particle count. Validation checks are implemented to ensure the input arrays have consistent shapes and the correct dimensionality.



In [2]:
import numpy as np
class NBodySystem:
    def __init__(self, positions, velocities):

        ''' Positions and velocities are N by 3 numpy arrays.'''

        if positions.shape[0] != velocities.shape[0]:
            raise ValueError(
                f"\n*** Different numbers of particle for position and velocity***")
        
        if positions.shape[1] != 3 or velocities.shape[1] != 3:
            raise ValueError(f"\n*** Positions or velocities are not 3 dimensional***")

        self.positions = np.array(positions, dtype=float)      
        self.velocities = np.array(velocities, dtype=float)  
        self.N = len(self.positions)


The simulation is carried out by the "simulate()" function, which advances the system forward in time while recording its full trajectory. At each step, the current positions and velocities are stored, after which the system state is updated in place via the chosen integration scheme.

A key feature is the clear separation between the system state, the time-stepping method ("step_function"), and the acceleration calculation ("acceleration_function"). This modular structure allows different integrators or acceleration solvers to be swapped in and out without modifying the core simulation loop, making the framework flexible.

The trajectory is stored in memory and animated later. This allows for animations without lag and simplifies investigations of how system properties, such as total energy, evolve with time.

The design feature of "**acceleration_kawargs" with the wrapper function "acceleration_function_with_arguments" allows a uniform interface between acceleration functions while enabling pass through of extra arguments if required. Initially this was intended to be an "approximation_order" for the Fast Multipole Method Algorithm, however due to time constraints this was only implemented for the "threshold" for the Barnes-Hut algorithm (described later). 

In [ ]:
def simulate(system,
            step_function,
            acceleration_function, 
            steps,
            dt=0.001,
            G=1, 
            softening=1e-5,
            **acceleration_kwargs):

    ''' Simulates and records trajectory of system.'''

    trajectory_positions = np.zeros((steps, system.N, 3))
    trajectory_velocities = np.zeros((steps, system.N, 3))

    def acceleration_function_with_arguments(sys):
        return acceleration_function(sys, G, softening, **acceleration_kwargs)

    for t in range(steps):
        trajectory_positions[t] = system.positions
        trajectory_velocities[t] = system.velocities

        step_function(system, acceleration_function_with_arguments, dt)
    
    return trajectory_positions, trajectory_velocities


### 3.2 Pairwise

The following python code was written to perform the pairwise simulation.

In [ ]:
def gravitational_vectorised_acceleration(system, G=1, softening=1e-5):

    ''' Naive approach with Numpy broadcasting.'''
    # ~320 times faster than Numpy loops.

    positions = system.positions

    # Numpy broadcasting instead of looping over index.
    dx = positions[:, np.newaxis, :] - positions[np.newaxis, :, :]   

    dist_sq = np.sum(dx**2, axis=2) + softening**2    
    inverse_distance_cubed = 1.0 / (dist_sq * np.sqrt(dist_sq))

    np.fill_diagonal(inverse_distance_cubed, 0)

    forces_div_masses = -G * dx * inverse_distance_cubed[:, :, np.newaxis] 
    acceleration = np.sum(forces_div_masses, axis=1)

    return acceleration

To implement this in Python, NumPy broadcasting was exploited in the "gravitational_vectorised_acceleration()" function. In the code below note the "dx = positions[:, np.newaxis, :] - positions[np.newaxis, :, :]" line. "positions[:, np.newaxis, :]" is an (N, 1, 3) array and positions[np.newaxis, :, :] is a (1, N, 3) array. As these shapes are compatible (though not the same) the result of the calculation is broadcast into the "dx" array of shape (N, N, 3) where "dx[i, j] = positions[i] - positions[j]"

Mathematically, this is equivalent to the following.

$$
\mathbf{dx} = \mathbf{1}\mathbf{x}^{T} - \mathbf{x}\mathbf{1}^{T} \tag{4},
\\
\text{where} \quad \mathbf{x} = \begin{pmatrix}
\vec{x_0}\\ \vec{x_1}\\ \vec{x_2}\\. \\ . \\. 
\end{pmatrix}
\\
\\
\text{and} \quad \mathbf{1} = \begin{pmatrix}
\vec{1}\\ \vec{1}\\ \vec{1}\\ . \\ . \\. 
\end{pmatrix}.
$$

All such operations are executed internally by NumPy in optimised C code, avoiding explicit Python loops and reducing overhead. As a result, this vectorised implementation is approximately 320 times faster than a naive Python loop-based approach (which can be found in "accelerations.py"). 

Self-interactions are removed by setting the diagonal elements of the inverse-distance term to zero via "np.fill_diagonal(inverse_distance_cubed, 0)", ensuring that particles do not exert forces on themselves.

Finally, the total acceleration on each particle is obtained by summing over all pairwise contributions, returning the expected array of shape (N, 3).

### 3.3 Barnes Hut
The Python implementation was the folowing


In [ ]:
class OctTreeNode:
    def __init__(self, centre, half_size):
        self.centre = centre
        self.half_size = half_size

        self.children = [None for i in range(8)]
        self.particle_index = None

        self.mass = 0
        self.centre_of_mass = np.zeros(3)
        


def get_octant(node, position):

    ''' Finds the octant of a particle. Stores position in bits as {xyz}. '''

    centre = node.centre
    index = 0
    if position[0] > centre[0]:
        index |= 1

    if position[1] > centre[1]: 
        index |= 2

    if position[2] > centre[2]:
        index |= 4

    return index
    

def create_child(node, octant):

    ''' Creates new node from old node.'''

    offset = np.array([
    0.5 if octant & 1 else -0.5,
    0.5 if octant & 2 else -0.5,
    0.5 if octant & 4 else -0.5
])

    new_centre = node.centre + (offset * node.half_size)
    return OctTreeNode(new_centre, node.half_size/2)
    

def insert(node, system, index):
    particle_position = system.positions[index]

    # Empty leaf.

    if node.particle_index is None and node.children[0] is None:
        node.particle_index = index
        return

    if node.children[0] is None:
        for i in range(8):
            node.children[i] = create_child(node, i)

        # Puts any exising particle into a child node. This is the recursive part.

        if node.particle_index is not None:
            old_index = node.particle_index
            node.particle_index = None
            insert(node, system, old_index)

    octant = get_octant(node, particle_position)
    insert(node.children[octant], system, index)


def compute_mass_distribution(node, system):

    ''' Finds mass, centre of mass of a node. '''

    # Empty node.
    if node is None:
        return 0.0, np.zeros(3)
    
    # Leaf node with particle.
    if node.particle_index is not None:
        node.mass = 1.0
        node.centre_of_mass = system.positions[node.particle_index]
        return node.mass, node.centre_of_mass
    
    # Internal node
    total_mass = 0.0
    com = np.zeros(3)

    for child in node.children:
        if child is None:
            continue
        m, c = compute_mass_distribution(child, system)
        total_mass += m
        com += m * c

    if total_mass > 0:
        com /= total_mass

    node.mass = total_mass
    node.centre_of_mass = com

    return total_mass, com
    

def compute_gravitational_acceleration(node, system, i, theta, G, softening):

    ''' Uses Barnes-Hut to find the gravitational acceleration on particle i due to a node. '''

    if node is None or node.mass == 0:
        return np.zeros(3)

    pos_i = system.positions[i]

   
    if node.particle_index == i:
        return np.zeros(3)

    r = node.centre_of_mass - pos_i
    dist_sq = np.dot(r, r) + softening**2
    dist = np.sqrt(dist_sq)

    s = node.half_size * 2

    
    if (node.particle_index is not None) or (s / dist < theta):
        inv_dist3 = 1.0 / (dist_sq * dist)
        return G * r * inv_dist3

    
    acc = np.zeros(3)
    for child in node.children:
        if child is not None:
            acc += compute_gravitational_acceleration(child, system, i, theta, G, softening)

    return acc


Note that this implementation differs slightly from the pseudocode, as node masses and centres of mass are computed in a separate pass via "compute_mass_distribution()" after the tree has been constructed using "insert()". This separation of responsibilities simplifies debugging.

The functions "get_octant()" and "create_child()" encode the particle’s position relative to the node centre as a three-bit index. Bitwise OR operations are used to construct this index, allowing the correct child node to be selected without multiple nested conditional IF statements.

### Fast Multipole Method
The code for this is shown below.

In [ ]:
import numpy as np

class OctTreeNode:
    def __init__(self, centre, half_size):
        self.centre     = np.asarray(centre, dtype=float)
        self.half_size  = float(half_size)
 
        self.children = [] 
        self.siblings = []
        self.cousins = []
        self.parent = None
        self.particles = []
        self.depth = 0
        self.interaction_list = []

        # Field from interior particles.
        # Second-order expansion, monopole dipole quarpole, 9 real coefficients
        # Stored as [q, dx, dy, dz, Qxx, Qyy, Qzz, Qxy, Qxz, Qyz]
        
        self.multipole = np.zeros(10)

        # Field from exterior particles.
        
        self.local = np.zeros(10)


def adjacent(node_a, node_b):
    '''
    Checks if two cells are touching (within 1e-10)
    '''
    threshold = 2.0 * node_a.half_size + 1e-10
    return all(abs(node_a.centre[d] - node_b.centre[d]) <= threshold for d in range(3))


def cousins(node):
    '''
    Fills in the cousin list for a node
    '''
    node.cousins = []
    parent = node.parent

    if parent is None:
        return
    

    for pibling in parent.siblings:
        for child in pibling.children:
            node.cousins.append(child)


def interaction_list(node):
    '''
    Fills in the interacton list from non-adjacent cousins
    '''
    for candidate in node.cousins:
        if adjacent(node, candidate) == False:
            node.interaction_list.append(candidate)


def contains(node, position):
    '''
    Checks if a particle is inside a node
    '''
    return all(
        abs(position[d] - node.centre[d]) <= node.half_size
        for d in range(3)
    )


def insert(particle_positions, index, node):
    '''
    Places particles into the node, recuring until it reaches a leaf
    '''
    if contains(node, particle_positions[index]) == False:
        return

    if len(node.children) == 0:
        node.particles.append(index)
        return

    for child in node.children:
        if contains(child, particle_positions[index]):
            insert(particle_positions, index, child)


def create_child(node):
    ''' 
    Creates 8 children from a node
    '''
    child_half_size = node.half_size / 2

    for i in range(8):

        offset = np.array([
        0.5 if i & 1 else -0.5,
        0.5 if i & 2 else -0.5,
        0.5 if i & 4 else -0.5
        ])

        child_centre = node.centre + offset * node.half_size
        child = OctTreeNode(child_centre, child_half_size)
        
        child.parent = node
        child.depth = node.depth + 1

        node.children.append(child)

    for child in node.children:
        child.siblings = [c for c in node.children if c is not child]
    

def tree(root_node, max_depth):
    '''
    Builds the Tree
    '''
    def recurse_create_child(node):


        if node.depth == max_depth:
            return

        create_child(node)

        for child in node.children:
            recurse_create_child(child)

    recurse_create_child(root_node)


def populate_tree(root_node, max_depth, particle_positions, particle_number):
    '''
    Inserts particles into the proper nodes, and fills in the node cousins and interaction list
    '''

    def recurse_cousins(node):

        if node.depth == max_depth:
            return
        
        cousins(node)

        # Recurse
        for child in node.children:
            recurse_cousins(child)

    recurse_cousins(root_node)

    def recurse_interaction_list(node):

        if node.depth == max_depth:
            return
        
        interaction_list(node)

        # Recurse
        for child in node.children:
            recurse_interaction_list(child)

    recurse_interaction_list(root_node)

    for i in range(particle_number):
        insert(particle_positions, i, root_node)



def compute_multipole_expansions(leaf_node, particle_positions):
    '''
    Computes 2nd order expansion of the potentisl field inside a leaf about its centre 
    due to the particles 
    '''
    q = 0.0
    p = np.zeros(3)
    Q = np.zeros((3, 3))

    for i in leaf_node.particles:
        r = particle_positions[i] - leaf_node.centre

        q +=1
        p += r
        Q +=  (3.0 * np.outer(r, r) - (np.dot(r, r) * np.eye(3)))

    leaf_node.multipole[0] = q
    leaf_node.multipole[1:4] = p
    leaf_node.multipole[4] = Q[0, 0]
    leaf_node.multipole[5] = Q[1, 1]
    leaf_node.multipole[6] = Q[2, 2]
    leaf_node.multipole[7] = Q[0, 1]
    leaf_node.multipole[8] = Q[0, 2]
    leaf_node.multipole[9] = Q[1, 2]



def compute_leaf_multipoles(root_node, max_depth, particle_positions):
    '''
    Goes through the tree and finds the multipole expansion of every single node
    '''
    def recurse(node):
        if node.depth < max_depth:
            for child in node.children:
                recurse(child)
            return

        compute_multipole_expansions(node, particle_positions)

    recurse(root_node)


def shift_multipole(source_node, target_node):
    '''
    Shifts multipole expansion from source_node centre to target_node centre.
    '''

   
    d = source_node.centre - target_node.centre

    q = source_node.multipole[0]
    p = source_node.multipole[1:4]
    Q = np.array([
        [source_node.multipole[4], source_node.multipole[7], source_node.multipole[8]],
        [source_node.multipole[7], source_node.multipole[5], source_node.multipole[9]],
        [source_node.multipole[8], source_node.multipole[9], source_node.multipole[6]]
    ])

  

    q_new = q

    p_new = p + q * d

    d_outer_d = np.outer(d, d)
    d_outer_p = np.outer(d, p)
    p_outer_d = np.outer(p, d)

    Q_new = (
        Q
        + 3.0 * (d_outer_p + p_outer_d)
        + 3.0 * q * d_outer_d
        - q * np.dot(d, d) * np.eye(3)
    )


    target_node.multipole[0] += q_new
    target_node.multipole[1:4] += p_new

    target_node.multipole[4] += Q_new[0, 0]
    target_node.multipole[5] += Q_new[1, 1]
    target_node.multipole[6] += Q_new[2, 2]
    target_node.multipole[7] += Q_new[0, 1]
    target_node.multipole[8] += Q_new[0, 2]
    target_node.multipole[9] += Q_new[1, 2]




def upwards_path(root_node, max_depth, particle_positions):
    '''
    Construct multipole expansion for leaf nodes and propagates them upwards 
    to the root. 
    '''
    compute_leaf_multipoles(root_node, max_depth, particle_positions)

    def recurse(node):

        if node.depth == max_depth:
            return

        # First recurse to children
        for child in node.children:
            recurse(child)

        node.multipole[:] = 0.0

        for child in node.children:
            shift_multipole(child, node)

    recurse(root_node)


def shift_multipole_to_local(source_node, target_node):
    '''
    Converts source multipole expansion into a local expansion at target node.
    (M2L operator, up to quadrupole order)
    '''

    R = target_node.centre - source_node.centre
    r = np.linalg.norm(R)

    q = source_node.multipole[0]
    p = source_node.multipole[1:4]
    Q = np.array([
        [source_node.multipole[4], source_node.multipole[7], source_node.multipole[8]],
        [source_node.multipole[7], source_node.multipole[5], source_node.multipole[9]],
        [source_node.multipole[8], source_node.multipole[9], source_node.multipole[6]]
    ])

    I = np.eye(3)

    # Precompute powers
    r2 = r * r
    r3 = r2 * r
    r5 = r3 * r2
    r7 = r5 * r2


    phi = (
        q / r
        + np.dot(p, R) / r3
        + 0.5 * np.sum(Q * (3 * np.outer(R, R) - r2 * I)) / r5
    )


    grad = (
        -q * R / r3
        + (p / r3 - 3 * np.dot(p, R) * R / r5)
        + (np.dot(Q,R) / r5 - 5 * (R @ Q @ R) * R / r7)
    )

    term1 = q * (3 * np.outer(R, R) - r2*I)/r5

    term2 = (
        3 * (np.outer(p,R) + np.outer(R,p))/r5
        - 15 * np.dot(p,R) * np.outer(R,R)/r7
    )

    term3 = (
        Q / r5
        - 5 * (np.outer(np.dot(Q, R), R) + np.outer(R, np.dot(Q, R))) / r7
    )

    H = term1 + term2 + term3


    target_node.local[0] += phi
    target_node.local[1:4] += grad

    target_node.local[4] += H[0, 0]
    target_node.local[5] += H[1, 1]
    target_node.local[6] += H[2, 2]
    target_node.local[7] += H[0, 1]
    target_node.local[8] += H[0, 2]
    target_node.local[9] += H[1, 2]


def shift_local(source_node, target_node):
    '''
    Shifts local expansion from source_node (parent) centre to target_node (child) centre.
    (L2L operator, up to quadrupole order)
    '''
    d = target_node.centre - source_node.centre

    phi  = source_node.local[0]
    grad = source_node.local[1:4]
    H    = np.array([
        [source_node.local[4], source_node.local[7], source_node.local[8]],
        [source_node.local[7], source_node.local[5], source_node.local[9]],
        [source_node.local[8], source_node.local[9], source_node.local[6]]
    ])

  
    phi_new = phi + np.dot(grad, d) + 0.5 * np.dot(d, np.dot(H, d))


    grad_new = grad + np.dot(H, d)

    H_new = H.copy()

    target_node.local[0] += phi_new
    target_node.local[1:4] += grad_new
    target_node.local[4] += H_new[0, 0]
    target_node.local[5] += H_new[1, 1]
    target_node.local[6] += H_new[2, 2]
    target_node.local[7] += H_new[0, 1]
    target_node.local[8] += H_new[0, 2]
    target_node.local[9] += H_new[1, 2]

def evaluate_local_expansion(leaf_node, particle_position):
    '''
    Returns the acceleration that a unit mass expriances at a point due to the potential
    '''

    r = particle_position - leaf_node.centre

    phi = leaf_node.local[0]
    g = leaf_node.local[1:4]

    H = np.array([
        [leaf_node.local[4], leaf_node.local[7], leaf_node.local[8]],
        [leaf_node.local[7], leaf_node.local[5], leaf_node.local[9]],
        [leaf_node.local[8], leaf_node.local[9], leaf_node.local[6]]
    ])

    grad = g + np.dot(H, r)
    acceleration = -grad

    return acceleration


def downward_pass(root_node, max_depth, particle_positions):
    '''
    Returns the acclereations of all the particles, found by evaluating the local 
    expansions of the leaf nodes at the particle positions, then performing particle-particle
    calculation between cousin nodes
    '''
    particle_accelerations = np.zeros_like(particle_positions)

    def recurse(node):

        
        for source in node.interaction_list:
            shift_multipole_to_local(source, node)

        if len(node.children) != 0:
            for child in node.children:
                shift_local(node, child)
                recurse(child)

        point_to_point_particles = []
        point_to_point_particles.extend(node.particles)

        for sibling in node.siblings:
            
            point_to_point_particles.extend(sibling.particles)
        
        # Point to point over adjacent cousins
        adjacent_cousins = [x for x in node.cousins if x not in node.interaction_list]
        for cousin in adjacent_cousins:
            point_to_point_particles.extend(cousin.particles)

        
        for particle_index in node.particles:
            
            local_acceleration = evaluate_local_expansion(node, particle_positions[particle_index])
            particle_accelerations[particle_index] += local_acceleration

            point_to_point_acceleration = np.zeros(3)
            
            for j in point_to_point_particles:
                if j != particle_index:

                    displacement = particle_positions[particle_index] - particle_positions[j]
                    abs_displacement = np.linalg.norm(displacement)
                    normalised_displacement = displacement / abs_displacement

                    acc = -normalised_displacement/ (abs_displacement ** 2)

                    point_to_point_acceleration += acc

            particle_accelerations[particle_index] += point_to_point_acceleration


    recurse(root_node)

    return particle_accelerations

From the init function, it is clear that the multipole expansion is hard-coded to the quadrupole level. An octupole expansion was explored, but its $O(P^{2})$ time complexity made the simulation prohibitively slow to run in practice.

The "populate_tree()" function is well separated from the rest of the programme, and an attempt was made to accelerate particle insertion using Morton codes. In practice, time constraints prevented a full implementation, though the partially completed work is available in "fast_multipole_morton_attempt.py"



# §4 Tests and Results

### 4.1 Time Complexity

To test the time complexity of the algorithms, the number of single-frame simulations of N particles that could be computed within a fixed time interval of two seconds was measured. This procedure was repeated multiple times, allowing the mean runtime and standard deviation to be determined. The experiment was carried out for a range of particle numbers $N = [10, 20, ..., 700, 1000]$, across three virialisation regimes, and for each algorithm.

The Barnes–Hut algorithm was additionally tested for different values of the opening angle parameter, $\theta = [0, 0.5, 1]$, to examine its effect on computational performance.


<img src="plots/time_complexity.png" width="100%">


**Fig. 12:** *Scaling of all algorithms across virialisation regimes. The runtime is normalised by $t_{0}$​, defined as the mean runtime at $N=10$. This allows for easier comparisons of scaling between algorithms. The lines of $N$ and $N^{2}$ have been included to compare to the theoretical slowest (pariwise) and quickest (FMM) algorithms.*

This figure shows that the virialisation regime does not seem to affect how the algorithms perform. This is actually expected. $r_{vir} = 2T/|U|$, where $U$ (GPE) depends on the length scale of the system. For every algorithm, Python stores all the length/positional data of the system as 64-bit floats. Calculations involving numbers of the same lengths occur in the same time, independent of the value of those numbers. This is why the plots look so similar across systems of vastly different Virial ratios.

To empirically investigate how the algorithms actually scale, best curves of $AN\log(N)$ and $AN^{b}$ were fit to the time complexity data for $r_{vir} = 1$. The functions $AN\log(N)$ and $AN^{b}$ were chosen as this is how the algorithms were expected to scale. The free parameters of $A,b$ were found via "scipy.optimise.curve_fit". The $R^{2}$ values were also found via dividing the residual sum of squares by the total variance of the data, then subtracting that ratio from one.

$$
R^{2} = 1-\frac{\sum (y_{i} - f(x_{i}))^{2}}{\sum(y_{i} - \bar{y} )^{2}},
$$

where $y_{i}$ and $x_{i}$ are the time and particle number of a data point, $\bar{y}$ is the mean time of all the measurements and $f$ is the function that scipy produces. 

<div style="text-align: center;">
    <img src="plots/virial1_Vectorised_Pairwise.png" width="70%">
</div>

**Fig. 13:** *Time complexity of the pairwise algorithm where $r_{vir} = 1$. The time has not been normalised*


Clearly the power-law fit is better than the logarithmic one. The power of $1.95$ is close to, but lower than, the expected value of $2$. This may be because the flatter region for $N< \sim 50$ drags the index down overall. This flatter section most likely exists because of some unexpected Numpy behviour.

<div style="text-align: center;">
    <img src="plots/bh_theta_fits.png" width="100%">
</div>

**Fig. 14** *Time complexity and best fits for the BH algorithms with varying values of $\theta$.*

At $\theta = 0$ the scaling is clearly a power law of index $2$. This is expected, as BH with a threshold of $\theta = 0$ reduces to the pairwise algorithm.

For $\theta = 0.5$ some approximations are made, leading to performance between $Nlog(N)$ and $N^{2}$. This is clearly shown as the power law fit now has index of $1.43$, and the fit of the $N\log(N)$ curve ($R^{2} = 0.987$) is better than that for $\theta = 0$.

At $\theta = 1$ there should be no direct particle-particle calculations and BH should scale like $N\log(N)$. While the $N\log(N)$ does have a much better fit than for the other values of $\theta$, it still has a lower $R^{2}$ than the power law. The $N\log(N)$ appears to fit better for large values of $N$ ($\sim 200+$) than for smaller values, so there may be some initialisation overhead. This may be why the power law has a better fit over this range, but as $N \rightarrow \infin$ the $N\log(N)$ scaling may still hold.

<div style="text-align: center;">
    <img src="plots/virial1_FMM.png" width="70%">
</div>

**Fig. 15:** *The Fast Multipole Method Algorithm time complexity with fitted curves.*

Neither of these functions fit particularly well to this plot. There appears to be a local maximum near $N=50$ with the slope varying across the range of $N$. This extra behaviour is likely to do with the tree having $\lfloor \frac{N}{8} \rfloor$ particle-containing leaf nodes. For values of $N$ just below a power of $8$ the leaf nodes become dense, leading to more direct particle - particle interactions being calculated slowing the algorithm down. With such poor fits for both curves, it is difficult to conclusively state that the algorithm scales as the expected $O(N)$, but the index of $1.19$ is lower than that of any other algorithm, suggesting that it might.

When deciding what algorithm to use for an $N$-body simulation the absolute run time must be considered. Lower order algorithms will always be faster as $N \rightarrow \infin$, but they may not be for smaller values.

<div style="text-align: center;">
    <img src="plots/bar_comparison.png" width="100%">
</div>

**Fig. 16:** *The absolute times it takes for one frame of a simulation to be generated, for $N = [10^{0}, 10^{3}]$. In this test $r_{Vir} = 1.$*

# §-1 References and Acknowledgements

**[1]**: J.A. Peacock, *Cosmological Physics*, 1st Edition, pp. 554 (1999)

**[2]**: T. Padmanabhan, *Structure Formation in the Universe*, 1st Edition, pp. 391 (1993)

**[3]**: G. Efstathiou, M. Davis, C.S. Frenk, S.D.M White, *Numerical Techniques for Large Cosmological N-Body Simulations*, Astrophysical Journal Supplement Series, Volume 57, pp. 241-260 (1985)

**[4]**: R. Pakmor et al., *The MillenniumTNG Project: The Hydrodynamical Full Physics Simulation and a First Look at its Galaxy Clusters*, Monthly Notices of the Royal Astronomical Society (2023)

**[5]**: J.Barnes, P.Hut, *A Hierarchical O(N log N) Force-Calculation Algorithm*, Nature, Volume 324, Issue 6096, pp. 446-449 (1986)

**[6]**: A. Appel, *An Efficient Program for Many-Body Simulation*, SIAM Journal on Scientific and Statistical Computing, Volume 6, Issue 1, pp. 85-103 (1985)

**[7]**: K. Esselink, *The order of Appel's algorithm*, Information Processing Letters, Volume 41, Issue 3, pp. 141-147 (1992) 

**[8]**: L. Greengard, V. Rokhlin Jr., *A Fast Algorithm for Particle Simulations*, Journal of Computational Physics, Volume 73, Issue 2, pp. 325-348 (1987)